# 🌱 Pflanzen- & Krankheitserkennung via Transfer Learning

Modul *Maschinelles Sehen*. Wir passen ein vortrainiertes CNN (ResNet18 / EfficientNet-B0) per **Transfer Learning** an den [PlantVillage]-Datensatz an.

**Ansatz:** ein kombiniertes Modell mit 38 Klassen der Form `Pflanze___Krankheit` (z. B. `Tomato___Late_blight`). Pflanze und Krankheit werden für die Ausgabe per `___`-Split getrennt.

> **In Colab:** `Runtime → Change runtime type → GPU` aktivieren!

[PlantVillage]: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset

## 1. Projektcode bereitstellen

Die Logik liegt in den `src/`-Modulen. In Colab entweder das Repo klonen **oder** den `src/`-Ordner per Datei-Upload hochladen. Diese Zelle deckt beide Fälle ab.

In [ ]:
import os, sys

# Variante A: eigenes Git-Repo klonen (URL anpassen)
# !git clone https://github.com/DEIN_USER/MS_Projekt.git
# PROJECT_ROOT = '/content/MS_Projekt'

# Variante B: lokal / src bereits vorhanden
PROJECT_ROOT = os.path.abspath('..') if os.path.isdir('../src') else os.path.abspath('.')

assert os.path.isdir(os.path.join(PROJECT_ROOT, 'src')), \
    f'src/ nicht gefunden unter {PROJECT_ROOT} – Repo klonen oder src/ hochladen.'
sys.path.insert(0, PROJECT_ROOT)
print('Projekt-Root:', PROJECT_ROOT)

## 2. Datensatz von Kaggle laden

Kaggle-Token (`kaggle.json`) über *Account → Create New API Token* herunterladen und unten hochladen. Überspringe diese Zelle, wenn der Datensatz schon lokal liegt.

In [ ]:
# --- Nur in Colab ausführen ---
# from google.colab import files
# files.upload()  # kaggle.json auswählen
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p data --unzip

# Pfad zum Datensatz (ImageFolder-Layout: train/<klasse>/.. , valid/<klasse>/..)
DATA_ROOT = os.path.join(
    PROJECT_ROOT, 'data',
    'New Plant Diseases Dataset(Augmented)',
    'New Plant Diseases Dataset(Augmented)'
)
# Falleback, falls der Datensatz schon woanders liegt:
if not os.path.isdir(DATA_ROOT):
    DATA_ROOT = os.path.join(PROJECT_ROOT, 'data')
print('Daten-Root:', DATA_ROOT)

## 3. Daten laden & ansehen

In [ ]:
import torch
import matplotlib.pyplot as plt

from src.data import create_dataloaders, split_label, IMAGENET_MEAN, IMAGENET_STD
from src.model import build_model
from src.train import train, get_device
from src.evaluate import evaluate, plot_confusion_matrix
from src.predict import predict_image

device = get_device()
print('Device:', device)

train_loader, val_loader, class_names = create_dataloaders(DATA_ROOT, batch_size=32)
print(f'{len(class_names)} Klassen, z. B.:', class_names[:5])

In [ ]:
# Ein paar Beispielbilder anzeigen (De-Normalisierung für die Darstellung)
import numpy as np

mean = np.array(IMAGENET_MEAN); std = np.array(IMAGENET_STD)
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, lbl in zip(axes.ravel(), images, labels):
    img = img.permute(1, 2, 0).numpy() * std + mean
    plant, disease = split_label(class_names[lbl])
    ax.imshow(np.clip(img, 0, 1)); ax.axis('off')
    ax.set_title(f'{plant}\n{disease}', fontsize=9)
plt.tight_layout(); plt.show()

## 4. Modell aufbauen

Vortrainiertes Backbone, neuer Klassifikationskopf mit `len(class_names)` Ausgängen. Backbone wird zunächst eingefroren (Feature-Extraction).

In [ ]:
BACKBONE = 'resnet18'  # oder 'efficientnet_b0'
model = build_model(num_classes=len(class_names), backbone=BACKBONE, freeze_backbone=True)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'Trainierbare Parameter (Phase A): {n_trainable:,} von {n_total:,}')

## 5. Training (Zwei-Phasen)

`train()` führt automatisch beide Phasen aus: erst nur den Kopf, dann das ganze Netz mit kleiner Lernrate. Das beste Modell (nach Val-Accuracy) wird unter `models/best_model.pth` gespeichert.

In [ ]:
model, class_names, history = train(
    data_root=DATA_ROOT,
    backbone=BACKBONE,
    epochs_head=3,
    epochs_finetune=5,
    batch_size=32,
    out_path=os.path.join(PROJECT_ROOT, 'models', 'best_model.pth'),
)

In [ ]:
# Lernkurven
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='train'); ax1.plot(history['val_loss'], label='val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoche'); ax1.legend()
ax2.plot(history['train_acc'], label='train'); ax2.plot(history['val_acc'], label='val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoche'); ax2.legend()
plt.tight_layout(); plt.show()

## 6. Evaluation

Accuracy, Per-Klassen-Report und Confusion Matrix auf dem Validierungs-Set.

In [ ]:
acc, y_true, y_pred = evaluate(model, val_loader, class_names, device=device)
fig = plot_confusion_matrix(y_true, y_pred, class_names, normalize=True)
plt.show()

## 7. Vorhersage auf einem Einzelbild

Ausgabe getrennt nach **Pflanze** und **Krankheit** – wie im Use-Case gewünscht.

In [ ]:
# Beispiel: erstes Bild aus dem Val-Loader (oder eigenen Pfad einsetzen)
images, labels = next(iter(val_loader))
img_tensor = images[0]
mean = np.array(IMAGENET_MEAN); std = np.array(IMAGENET_STD)
from PIL import Image
img_disp = np.clip(img_tensor.permute(1, 2, 0).numpy() * std + mean, 0, 1)
pil = Image.fromarray((img_disp * 255).astype('uint8'))

best, topk = predict_image(pil, model, class_names, device=device, top_k=3)
plt.imshow(img_disp); plt.axis('off'); plt.title(str(best)); plt.show()
print('Wahr:', class_names[labels[0]])
for label, prob in topk:
    print(f'  {label:40s} {prob*100:5.1f}%')

## 8. Nächste Schritte

- **Realitäts-Check:** eigenes Handyfoto durch `predict_image` schicken – PlantVillage sind Laboraufnahmen, echte Fotos sind schwieriger.
- **Backbone-Vergleich:** `BACKBONE = 'efficientnet_b0'` setzen und Accuracy/Zeit vergleichen.
- **Grad-CAM:** visualisieren, worauf das Netz achtet.
- **Multi-Task:** zwei getrennte Köpfe für Pflanze und Krankheit als Architektur-Variante.